# Reviewer comment 5 — evaluating the framework on Deepfake-Eval-2024

Reviewer 5: *"The near-perfect accuracy on PolyGlotFake may be due to dataset-specific
artifacts... The authors should test the framework on more diverse in-the-wild datasets
like DeepfakeEval-2024 to demonstrate real-world generalization."*

**Read this section before running anything — access is gated.**

Deepfake-Eval-2024 (Chandra et al., CVPRW 2026; arXiv:2503.02857) is released under
CC-BY-SA-4.0 but **access is gated**: you must request it with an institutional or company
email and give evidence that you work on deepfake detection. Approval is not instant, so
**request access before you need the results**. The gate is documented in the paper itself
(Section 6 / Appendix A.5).

**The terms of use matter for what you can claim.** The Hugging Face dataset card states the
data is for *evaluation only* and that training on it goes against the terms. This notebook
therefore runs the released checkpoints **zero-shot** and never finetunes. That is also the
scientifically right choice here: the reviewer's concern is about generalisation, and
finetuning on the benchmark would answer a different question.

**What this notebook produces.** `deepfake_eval_2024_results.json` — zero-shot AUC, accuracy,
precision, recall and F1 for the five-agent framework on the audio-visual subset, plus the
Phase-2 escalation rate (which the manuscript argues is a distribution-shift indicator, so
this is a direct test of that claim).

**Context for interpreting whatever number you get.** On this benchmark, published open-source
detectors collapse: average AUC drops of 50% (video), 48% (audio) and 45% (image) against the
academic sets they were originally tested on, and **no off-the-shelf open-source model exceeds
AUC 0.58**. Most directly relevant to this manuscript, **GenConViT — which we benchmark at
97.3% on PolyGlotFake — scores AUC 0.63 here, against 0.96 on its own test data**. The two
runnable open-source *multimodal* detectors score AUC 0.58 (AVF) and 0.42 (FGI). An AUC in the
0.6-0.7 range would therefore be a competitive in-the-wild result, not a poor one.

---


## Step 1 — Environment


In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
%cd /content
!git clone https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo 2>/dev/null || echo 'already cloned'
%cd /content/repo
!pip -q install -r requirements.txt
!pip -q install albumentations speechbrain timm librosa opencv-python-headless \
                facenet-pytorch datasets huggingface_hub moviepy


## Step 2 — Authenticate and pull the benchmark

Request access first at the Hugging Face dataset page, wait for approval, then create a read
token. If the cell below 403s, your request has not been granted yet.


In [ ]:
from huggingface_hub import notebook_login, snapshot_download
notebook_login()


In [ ]:
import os
REPO_ID = 'nuriachandra/Deepfake-Eval-2024'   # confirm against the paper's release page
DEST = '/content/drive/MyDrive/deepfake_eval_2024'
try:
    path = snapshot_download(repo_id=REPO_ID, repo_type='dataset', local_dir=DEST,
                             allow_patterns=['*video*', '*.csv', '*.json'])
    print('downloaded to', path)
except Exception as e:
    print('DOWNLOAD FAILED:', type(e).__name__, e)
    print('\nIf this is a 403/GatedRepoError your access request is still pending.')
    print('If the repo id is wrong, find the current one from the paper or TrueMedia.org.')


## Step 3 — Select the audio-visual subset

The framework needs **both** streams: three of its five agents are audio or cross-modal. The
benchmark's video split is 2,036 files (1,072 real / 964 fake, 45.1 h) and the paper states
the majority of video data has corresponding labelled audio. Keep only clips that actually
carry a decodable audio track, and record how many were dropped — that count belongs in the
manuscript, because it defines what the reported number covers.


In [ ]:
import pandas as pd, glob, os, subprocess, json

meta = None
for c in glob.glob(f'{DEST}/**/*.csv', recursive=True):
    df = pd.read_csv(c)
    if any('label' in x.lower() for x in df.columns):
        meta = df; print('metadata:', c, df.shape, list(df.columns)[:12]); break
assert meta is not None, 'no labelled metadata csv found - inspect the download'
display(meta.head())


In [ ]:
def has_audio(path):
    r = subprocess.run(['ffprobe','-v','error','-select_streams','a',
                        '-show_entries','stream=codec_type','-of','csv=p=0', path],
                       capture_output=True, text=True)
    return 'audio' in r.stdout

# EDIT: point these at the actual column names in `meta`
PATH_COL, LABEL_COL = 'filename', 'ground_truth'

rows, dropped_no_audio, dropped_missing = [], 0, 0
for _, r in meta.iterrows():
    cand = glob.glob(f'{DEST}/**/{os.path.basename(str(r[PATH_COL]))}', recursive=True)
    if not cand: dropped_missing += 1; continue
    if not has_audio(cand[0]): dropped_no_audio += 1; continue
    rows.append({'path': cand[0], 'label': str(r[LABEL_COL]).strip().lower()})

av = pd.DataFrame(rows)
av = av[av.label.isin(['real','fake'])].reset_index(drop=True)
print(f'audio-visual subset: {len(av)} clips  ({(av.label=="real").sum()} real, {(av.label=="fake").sum()} fake)')
print(f'dropped: {dropped_no_audio} with no audio track, {dropped_missing} file not found')
av.to_csv('/content/av_subset.csv', index=False)


## Step 4 — Preprocess into the framework's input format

Identical preprocessing to PolyGlotFake, which is what makes the comparison fair: MTCNN faces
at confidence 0.95 with frame stride 10, audio via librosa at 16 kHz standardised to 5 s.

**Note the output directory.** `src/orchestrator.py` takes no command-line arguments: it reads
`CONFIG["data_dir"]` and looks specifically in its `test/` subdirectory. Writing the clips
straight into `dfe2024_root/test/` lets Step 5 point the orchestrator at them with a symlink,
with no edit to the source.

In-the-wild footage is messier than PolyGlotFake, so expect failures — clips with no
detectable face, multiple faces, or corrupt audio. **Count them and keep the count**: a
framework that silently drops a third of a benchmark is not reporting the benchmark.


In [ ]:
import numpy as np, librosa, cv2, torch, os, json
from facenet_pytorch import MTCNN
from tqdm.auto import tqdm

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(keep_all=False, device=DEV, thresholds=[0.6, 0.7, 0.95])
OUT = '/content/dfe2024_root/test'          # note the 'test' subdirectory
os.makedirs(OUT, exist_ok=True)

def preprocess(path, label, stride=10, max_frames=20):
    cap = cv2.VideoCapture(path); faces, i = [], 0
    while len(faces) < max_frames:
        ok, fr = cap.read()
        if not ok: break
        if i % stride == 0:
            f = mtcnn(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
            if f is not None: faces.append(f.cpu().numpy())
        i += 1
    cap.release()
    if not faces: raise ValueError('no face detected')
    y, _ = librosa.load(path, sr=16000, mono=True)
    y = np.pad(y, (0, max(0, 80000 - len(y))))[:80000]
    if not np.any(y): raise ValueError('silent or unreadable audio')
    mel = librosa.feature.melspectrogram(y=y, sr=16000, n_mels=224)
    return dict(faces=np.stack(faces), audio=y.astype(np.float32),
                mel=librosa.power_to_db(mel).astype(np.float32),
                label=1 if label == 'fake' else 0)

ok_n, fail = 0, {}
for _, r in tqdm(av.iterrows(), total=len(av)):
    # the orchestrator infers ground truth from the _label_{real,fake} filename suffix
    stem = os.path.splitext(os.path.basename(r.path))[0]
    dst = os.path.join(OUT, f'{stem}_label_{r.label}.npz')
    if os.path.exists(dst): ok_n += 1; continue
    try:
        np.savez_compressed(dst, **preprocess(r.path, r.label)); ok_n += 1
    except Exception as e:
        fail[str(e)[:60]] = fail.get(str(e)[:60], 0) + 1

N_USABLE = ok_n
print(f'\nusable: {ok_n} / {len(av)}   ({100*ok_n/max(len(av),1):.1f}%)')
print('failure reasons:', json.dumps(fail, indent=2))
json.dump({'attempted': int(len(av)), 'usable': int(ok_n), 'failures': fail},
          open('/content/dfe2024_coverage.json', 'w'), indent=2)


## Step 5 — Point the orchestrator at the benchmark, then run it zero-shot

**`src/orchestrator.py` accepts no arguments.** It hard-codes
`data_dir = "data/polyglot_processed_all_unbalanced"`, reads its `test/` subdirectory, and
writes `analysis_results_with_5_agents.csv` into the working directory. Passing
`--data_dir` or `--output_file` does not error — the flags are silently ignored, and the run
would evaluate whatever happens to sit at the hard-coded path. That is the dangerous failure
mode here, because it produces a plausible-looking number for the wrong dataset. The symlink
below redirects the hard-coded path, and Step 6 asserts the row count matches this benchmark.

Released checkpoints, unchanged. Decision threshold τ = 0.37 and disagreement threshold
τ_disagree = 0.30 exactly as in the paper — **do not retune them on this benchmark**, or the
result stops being a generalisation test.


In [ ]:
import os, pathlib, shutil
os.chdir('/content/repo')

link = pathlib.Path('data/polyglot_processed_all_unbalanced')
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.exists():
    (link.unlink if link.is_symlink() or link.is_file() else shutil.rmtree)(link)
link.symlink_to('/content/dfe2024_root')

test_dir = 'data/polyglot_processed_all_unbalanced/test'
n = len(os.listdir(test_dir))
print(f'{test_dir} -> {os.path.realpath(test_dir)}  ({n} clips)')
assert n == N_USABLE, f'symlink sees {n} clips, preprocessing wrote {N_USABLE}'

# a stale PolyGlotFake result here would be silently overwritten or, worse, re-read
prev = 'analysis_results_with_5_agents.csv'
if os.path.exists(prev): os.remove(prev); print('removed a pre-existing', prev)


In [ ]:
!python src/orchestrator.py 2>&1 | tail -25


## Step 6 — Metrics, plus the escalation-rate test

The first cell asserts the run actually covered this benchmark rather than a leftover
PolyGlotFake tree — 2,162 rows would mean the symlink did not take effect.

The escalation figure is the interesting one. The manuscript claims the Phase-2 escalation
rate acts as a built-in distribution-shift indicator (8.8% on PolyGlotFake, 69.4% on the
YouTube benchmark). Deepfake-Eval-2024 is a harder shift than YouTube, so escalation should
be high. If it is, that is independent evidence for the claim; if it is not, that is a finding
the revision should report honestly.


In [ ]:
import pandas as pd, numpy as np, json
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

df = pd.read_csv('/content/repo/analysis_results_with_5_agents.csv')
assert len(df) != 2162, ('2,162 rows means the orchestrator read the PolyGlotFake test set, '
                         'not this benchmark - the symlink in Step 5 did not take effect')
assert len(df) == N_USABLE, f'{len(df)} rows but {N_USABLE} clips were preprocessed'
print(f'{len(df)} rows, matches the {N_USABLE} preprocessed clips')

y = (df.ground_truth.str.lower() == 'fake').astype(int).to_numpy()
s = df.final_score.to_numpy(); TAU = 0.37
pred = (s > TAU).astype(int)
res = {'n': int(len(df)), 'n_real': int((y == 0).sum()), 'n_fake': int(y.sum()), 'tau': TAU,
       'auc': float(roc_auc_score(y, s)), 'ap': float(average_precision_score(y, s)),
       'accuracy': float(accuracy_score(y, pred)),
       'precision': float(precision_score(y, pred, zero_division=0)),
       'recall': float(recall_score(y, pred, zero_division=0)),
       'f1': float(f1_score(y, pred, zero_division=0)),
       'confusion_matrix': confusion_matrix(y, pred).tolist()}
if 'phase' in df.columns:
    res['escalation_rate'] = float(df.phase.astype(str)
                                   .str.contains('2|iterative', case=False).mean())
print(json.dumps(res, indent=2))
json.dump(res, open('/content/deepfake_eval_2024_results.json', 'w'), indent=2)


In [ ]:
# Put the result next to the published baselines from the benchmark paper (Table 3, S11)
ref = pd.DataFrame([
  ('GenConViT',        'video',      0.63, 0.96),
  ('FTCN',             'video',      0.50, 0.87),
  ('Styleflow',        'video',      0.51, 0.95),
  ('AASIST',           'audio',      0.43, 1.00),
  ('RawNet2',          'audio',      0.53, 0.99),
  ('P3 (Wang et al.)', 'audio',      0.58, 1.00),
  ('AVF',              'multimodal', 0.58, 0.945),
  ('FGI',              'multimodal', 0.42, 0.845),
], columns=['Model', 'Modality', 'AUC on Deepfake-Eval-2024', 'AUC on original benchmark'])
ref.loc[len(ref)] = ['Ours (5-agent)', 'multimodal', round(res['auc'], 3), 1.000]
display(ref.sort_values('AUC on Deepfake-Eval-2024', ascending=False))
ref.to_csv('/content/dfe2024_comparison.csv', index=False)


## Step 7 — Download


In [ ]:
from google.colab import files
for f in ['/content/deepfake_eval_2024_results.json', '/content/dfe2024_comparison.csv',
          '/content/dfe2024_coverage.json', '/content/dfe2024_5agent.csv']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)


---
## If access is not granted in time

This is a legitimate outcome and the revision already handles it. The cover letter states that
access is gated, cites the benchmark's own finding that open-source detectors lose ~50% AUC on
it, and points to the manuscript's existing YouTube in-the-wild evaluation (79.59% accuracy,
escalation rising 8.8% -> 69.4%) as the generalisation evidence that *is* in hand. Section 5.1
and the Conclusions both flag Deepfake-Eval-2024 as the next evaluation. Do not report a number
you did not measure.
